# LLMs for Business Analytics

Starter notebook for exploring LLMs with Python.

In [4]:
import os

import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from pathlib import Path

DATA_DIR = Path("datasets")
df = pd.read_csv(DATA_DIR / "ESGenius_w_ref_1136q.csv")
df.head()

,query_id,new_id,query,answer,A,B,C,D,Z,ref_page,ref_doc,source_text
0,831,1,"According to the IPCC AR6 Synthesis Report, wh...",A,Implementing targeted management strategies fo...,Failing to rebuild overexploited fisheries whi...,Limiting global warming but neglecting land re...,Prioritizing disaster risk management and earl...,Not sure,46,IPCC AR6 SYR.pdf,30 Summary for Policymakers Summary for Policy...
1,889,2,According to Climate Change 2023 — Synthesis R...,B,The geographical patterns of climatic impacts ...,Geographical patterns of climatic impacts for ...,The timing of reaching a specific GWL determin...,Global warming levels are defined exclusively ...,Not sure,80-81,IPCC AR6 SYR.pdf,64 Section 2 Section 1Section 2Global Warming ...
2,892,3,Which statement accurately reflects the relati...,C,High or very high confidence in attribution is...,The text explicitly states that medium confide...,Attribution of observed physical climate chang...,Increases in heavy precipitation and sea level...,Not sure,23,IPCC AR6 SYR.pdf,7 Summary for PolicymakersSummary for Policyma...
3,983,4,According to the Climate Change 2023 — Synthes...,D,Europe,Small Islands,North America,Asia,Not sure,92,IPCC AR6 SYR.pdf,76 Section 3 Section 1Section 3011.5234 011.52...
4,1077,5,According to the Climate Change 2023 — Synthes...,A,"Early warning systems, flood-proofing of build...","Afforestation, reforestation, and levees.","Agroecological practices, disaster risk manage...","Heat Health Action Plans, vector-borne disease...",Not sure,72,IPCC AR6 SYR.pdf,"56 Section 2 Section 1Section 2wetlands, range..."


## Perturbation Engine

Generates corrupted variants of both datasets to measure LLM robustness.  
Ground truth (`answer` / `Answer`) is **never modified** — only the input context is perturbed.

| Dataset | Perturbation types | Levels | Items |
|---|---|---|---|
| ESGenius | numeric, missing, label, format | 1–3 | 1 136 × 13 = 14 768 |
| Climate Finance Bench | numeric, missing, entity, format | 1–3 | 330 × 13 = 4 290 |

In [5]:
import re
import numpy as np
import pandas as pd
import nltk
from pathlib import Path

nltk.download('punkt_tab', quiet=True)
np.random.seed(42)

DATA_DIR = Path("datasets")

In [6]:
# ── Shared perturbation utilities (used by both datasets) ────────────────────

def missing_perturb(text, level):
    """Drop 10 / 25 / 50 % of sentences at random."""
    drop_rate = {1: 0.10, 2: 0.25, 3: 0.50}[level]
    sents = nltk.sent_tokenize(str(text))
    kept = [s for s in sents if np.random.random() > drop_rate]
    return ' '.join(kept) if kept else sents[0]  # always keep at least one sentence


def format_perturb(text, level):
    """Level 1: shuffle sentences. Level 2: + lowercase. Level 3: + strip punctuation."""
    sents = nltk.sent_tokenize(str(text))
    if level >= 1:
        np.random.shuffle(sents)
    text = ' '.join(sents)
    if level >= 2:
        text = text.lower()
    if level >= 3:
        text = re.sub(r'[^\w\s]', '', text)
    return text

### ESGenius — MCQ perturbations

Four perturbation types applied to each of the 1 136 items:
- **numeric** — randomly reassign each confidence phrase to a different level (level 1: one step up/down; levels 2–3: any other level) and multiply every number by a random factor (±10 / 25 / 50 %)
- **missing** — drop 10 / 25 / 50 % of sentences from `source_text`
- **label** — swap option texts (A/B/C/D); the answer key `answer` is never changed
- **format** — shuffle sentence order → lowercase → remove punctuation from `source_text`

In [7]:
esg = pd.read_csv(DATA_DIR / "ESGenius_w_ref_1136q.csv")

# ── ESGenius-specific perturbation functions ──────────────────────────────────

# Only the confidence phrases that actually appear in the ESGenius source_text.
# When a phrase is found, we randomly pick any OTHER level from this same list.
_CONFIDENCE_LEVELS = [
    'virtually certain',
    'very high confidence',
    'high confidence',
    'medium confidence',
    'low confidence',
]
_CONF_PATTERN = '|'.join(re.escape(c) for c in sorted(_CONFIDENCE_LEVELS, key=len, reverse=True))


def numeric_perturb_esg(text, level):
    """
    For each confidence phrase found, randomly pick a different one from the list.
      Level 1 (mild)     — only step one level up or down
                           e.g. 'high confidence' → 'very high confidence' or 'medium confidence'
      Level 2 (moderate) — any other confidence level from the list
      Level 3 (severe)   — any other confidence level from the list

    Then multiply every number in the text by (1 + U(-noise, +noise)):
      Level 1 → ±10%   e.g. 12.8 becomes anywhere from 11.5 to 14.1
      Level 2 → ±25%   e.g. 12.8 becomes anywhere from  9.6 to 16.0
      Level 3 → ±50%   e.g. 12.8 becomes anywhere from  6.4 to 19.2
    """
    def random_confidence(m):
        original = m.group().lower()
        if original not in _CONFIDENCE_LEVELS:
            return m.group()
        idx = _CONFIDENCE_LEVELS.index(original)
        if level == 1:
            neighbours = []
            if idx > 0:
                neighbours.append(_CONFIDENCE_LEVELS[idx - 1])
            if idx < len(_CONFIDENCE_LEVELS) - 1:
                neighbours.append(_CONFIDENCE_LEVELS[idx + 1])
            return np.random.choice(neighbours)
        else:
            candidates = [c for c in _CONFIDENCE_LEVELS if c != original]
            return np.random.choice(candidates)

    text = re.sub(_CONF_PATTERN, random_confidence, str(text), flags=re.IGNORECASE)

    noise = {1: 0.10, 2: 0.25, 3: 0.50}[level]
    return re.sub(
        r'\b\d+\.?\d*\b',
        lambda m: f"{float(m.group()) * (1 + np.random.uniform(-noise, noise)):.1f}",
        text,
    )


def label_perturb_esg(row, level):
    """
    Swap the text of answer options without changing the answer key.
    Level 1: swap correct <-> one wrong option.
    Level 2: rotate correct -> wrong[0] -> wrong[1] -> correct.
    Level 3: shuffle all four option texts.
    """
    opts = ['A', 'B', 'C', 'D']
    correct = row['answer']
    wrong = [o for o in opts if o != correct]
    new = row.copy()
    if level == 1:
        new[correct], new[wrong[0]] = row[wrong[0]], row[correct]
    elif level == 2:
        new[correct], new[wrong[0]], new[wrong[1]] = row[wrong[0]], row[wrong[1]], row[correct]
    elif level == 3:
        shuffled = [row[o] for o in opts]
        np.random.shuffle(shuffled)
        for o, t in zip(opts, shuffled):
            new[o] = t
    return new


# ── Generate all perturbed variants ──────────────────────────────────────────

rows = []
for _, row in esg.iterrows():
    base = row.copy()
    base['perturbation_type'] = 'none'
    base['perturbation_level'] = 0
    rows.append(base)

    for level in [1, 2, 3]:
        # numeric
        r = row.copy()
        r['source_text'] = numeric_perturb_esg(row['source_text'], level)
        r['perturbation_type'] = 'numeric'
        r['perturbation_level'] = level
        rows.append(r)

        # missing
        r = row.copy()
        r['source_text'] = missing_perturb(row['source_text'], level)
        r['perturbation_type'] = 'missing'
        r['perturbation_level'] = level
        rows.append(r)

        # label — answer key column is unchanged
        r = label_perturb_esg(row, level)
        r['perturbation_type'] = 'label'
        r['perturbation_level'] = level
        rows.append(r)

        # format
        r = row.copy()
        r['source_text'] = format_perturb(row['source_text'], level)
        r['perturbation_type'] = 'format'
        r['perturbation_level'] = level
        rows.append(r)

esg_perturbed = pd.DataFrame(rows)
esg_perturbed.to_csv(DATA_DIR / "esgenius_perturbed.csv", index=False)

print(f"ESGenius: {len(esg)} original x 13 variants = {len(esg_perturbed)} rows")
esg_perturbed[['new_id', 'answer', 'perturbation_type', 'perturbation_level', 'source_text']].head(14)

ESGenius: 1136 original x 13 variants = 14768 rows


,new_id,answer,perturbation_type,perturbation_level,source_text
0,1,A,none,0,30 Summary for Policymakers Summary for Policy...
0,1,A,numeric,1,27.9 Summary for Policymakers Summary for Poli...
0,1,A,missing,1,30 Summary for Policymakers Summary for Policy...
0,1,A,label,1,30 Summary for Policymakers Summary for Policy...
0,1,A,format,1,Land restoration contributes to climate change...
0,1,A,numeric,2,22.7 Summary for Policymakers Summary for Poli...
0,1,A,missing,2,30 Summary for Policymakers Summary for Policy...
0,1,A,label,2,30 Summary for Policymakers Summary for Policy...
0,1,A,format,2,different contexts include but are not limited...
0,1,A,numeric,3,20.0 Summary for Policymakers Summary for Poli...


### Climate Finance Bench — free-text perturbations

Four perturbation types applied to each of the 330 items (`label` replaced by `entity` — no MCQ options):
- **numeric** — jitter all numbers in `Document extracts` by ±10 / 25 / 50 %
- **missing** — drop 10 / 25 / 50 % of sentences from `Document extracts`
- **entity** — swap domain-specific labels according to a reproducible taxonomy (see cell below)
- **format** — shuffle sentence order → lowercase → remove punctuation

Ground truth `Answer` is never modified; scoring uses numeric extraction (PE/NR) or LLM-as-judge (LR).

#### Entity perturbation taxonomy

Seven entity types cover the labels that appear in any ESG sustainability report.
Levels control which types are active — cumulatively:

| Level | Active entity types |
|---|---|
| 1 | scope label, fiscal year |
| 2 | + emission unit, currency |
| 3 | + emission qualifier, magnitude unit, Scope 2 accounting method |

Each type has three components:
1. **pattern** — regex that detects it in any text
2. **pool** — the finite set of valid values for that type
3. **rule** — how the replacement is chosen (`random_other`: uniform draw from pool excluding the original; `shift_year`: shift the detected year by ±level years)

| Entity type | Example match | Pool |
|---|---|---|
| `scope_label` | "Scope 1 & 2" | Scope 1, Scope 2, Scope 3, Scope 1 & 2, Scope 1, 2 & 3 |
| `fiscal_year` | "FY2024" | any year ± 1/2/3 years |
| `emission_unit` | "MtCO2e" | tCO2e, ktCO2e, MtCO2e, GtCO2e |
| `currency` | "RMB" | RMB, USD, EUR, GBP, JPY |
| `emission_qualifier` | "net emissions" | net emissions, gross emissions |
| `magnitude_unit` | "million tons" | billion tons, million tons, thousand tons |
| `scope2_method` | "market-based" | market-based, location-based |

In [8]:
cfb = pd.read_excel(DATA_DIR / "Climate Finance Bench - Dataset.xlsx")

# ── Entity perturbation taxonomy ──────────────────────────────────────────────
#
# Each entry defines one ENTITY TYPE that can appear in any ESG report.
# Fields:
#   desc    – plain-English label (for documentation and the paper)
#   pattern – compiled regex that detects this entity in free text
#   pool    – explicit finite set of replacement candidates
#             (None = computed at runtime, see rule)
#   rule    – replacement strategy:
#             'random_other' : draw uniformly from pool, never the matched value
#             'shift_year'   : shift the matched year by ±level years

_ENTITY_TAXONOMY = {
    'scope_label': {
        'desc': 'GHG accounting boundary (Scope 1, 2, 3)',
        # Matches compound forms first (greedy optional groups), then single scopes
        'pattern': re.compile(
            r'\bScopes?\s+1(?:\s*[,&+]\s*2(?:\s*[,&+]\s*3)?)?\b'
            r'|\bScopes?\s+2\b'
            r'|\bScopes?\s+3\b',
            re.IGNORECASE,
        ),
        'pool': ['Scope 1', 'Scope 2', 'Scope 3', 'Scope 1 & 2', 'Scope 1, 2 & 3'],
        'rule': 'random_other',
    },
    'fiscal_year': {
        'desc': 'Reporting year reference (FY2024, FY2023, ...)',
        'pattern': re.compile(r'\bFY\s*(\d{4})\b', re.IGNORECASE),
        'pool': None,   # year is shifted dynamically
        'rule': 'shift_year',
    },
    'emission_unit': {
        'desc': 'GHG quantity unit — magnitude prefix and gas type',
        'pattern': re.compile(r'\b(?:G|M|k)?tCO2e?\b'),
        'pool': ['tCO2e', 'ktCO2e', 'MtCO2e', 'GtCO2e'],
        'rule': 'random_other',
    },
    'currency': {
        'desc': 'Currency denomination (RMB, USD, EUR, ...)',
        'pattern': re.compile(r'\b(?:RMB|CNY|USD|EUR|GBP|JPY|AUD|CAD|SGD)\b'),
        'pool': ['RMB', 'USD', 'EUR', 'GBP', 'JPY'],
        'rule': 'random_other',
    },
    'emission_qualifier': {
        'desc': 'Net vs gross emissions distinction',
        'pattern': re.compile(r'\b(?:net|gross)\s+emissions?\b', re.IGNORECASE),
        'pool': ['net emissions', 'gross emissions'],
        'rule': 'random_other',
    },
    'magnitude_unit': {
        'desc': 'Order-of-magnitude unit for mass quantities',
        'pattern': re.compile(
            r'\b(?:billion|million|thousand)\s+(?:metric\s+)?tons?\b', re.IGNORECASE
        ),
        'pool': ['billion tons', 'million tons', 'thousand tons'],
        'rule': 'random_other',
    },
    'scope2_method': {
        'desc': 'Scope 2 electricity accounting method',
        'pattern': re.compile(r'\b(?:market[- ]based|location[- ]based)\b', re.IGNORECASE),
        'pool': ['market-based', 'location-based'],
        'rule': 'random_other',
    },
}

# Which entity types are active at each level (cumulative)
_ENTITY_LEVELS = {
    1: ['scope_label', 'fiscal_year'],
    2: ['scope_label', 'fiscal_year', 'emission_unit', 'currency'],
    3: ['scope_label', 'fiscal_year', 'emission_unit', 'currency',
        'emission_qualifier', 'magnitude_unit', 'scope2_method'],
}


def numeric_perturb_cfb(text, level):
    """Jitter every numeric token in the extract by +/- noise %."""
    noise = {1: 0.10, 2: 0.25, 3: 0.50}[level]
    return re.sub(
        r'\b\d+\.?\d*\b',
        lambda m: f"{float(m.group()) * (1 + np.random.uniform(-noise, noise)):.1f}",
        str(text),
    )


def entity_perturb(text, level):
    """
    Replace domain-specific entities using the taxonomy in _ENTITY_TAXONOMY.

    For each active entity type at this level, every matched instance is
    replaced with a value drawn uniformly at random from that type's pool
    (never the original value).

    Fiscal years are shifted by ±level years (direction chosen randomly).
    All other replacements are drawn from the explicit pool defined above.
    Reproducibility is guaranteed by np.random.seed(42) at notebook startup.
    """
    result = str(text)

    for entity_type in _ENTITY_LEVELS[level]:
        defn = _ENTITY_TAXONOMY[entity_type]

        if defn['rule'] == 'random_other':
            pool = defn['pool']
            def replace_entity(m, pool=pool):
                original = m.group().lower()
                candidates = [p for p in pool if p.lower() != original]
                return np.random.choice(candidates) if candidates else m.group()
            result = defn['pattern'].sub(replace_entity, result)

        elif defn['rule'] == 'shift_year':
            shift = level
            def replace_year(m, shift=shift):
                year = int(m.group(1))
                direction = np.random.choice([-1, 1])
                return f"FY{year + direction * shift}"
            result = defn['pattern'].sub(replace_year, result)

    return result


# ── Generate all perturbed variants ──────────────────────────────────────────

EXTRACT_COL = 'Document extracts'
rows = []

for _, row in cfb.iterrows():
    base = row.copy()
    base['perturbation_type'] = 'none'
    base['perturbation_level'] = 0
    rows.append(base)

    for level in [1, 2, 3]:
        # numeric
        r = row.copy()
        r[EXTRACT_COL] = numeric_perturb_cfb(row[EXTRACT_COL], level)
        r['perturbation_type'] = 'numeric'
        r['perturbation_level'] = level
        rows.append(r)

        # missing
        r = row.copy()
        r[EXTRACT_COL] = missing_perturb(row[EXTRACT_COL], level)
        r['perturbation_type'] = 'missing'
        r['perturbation_level'] = level
        rows.append(r)

        # entity — answer column is never touched
        r = row.copy()
        r[EXTRACT_COL] = entity_perturb(row[EXTRACT_COL], level)
        r['perturbation_type'] = 'entity'
        r['perturbation_level'] = level
        rows.append(r)

        # format
        r = row.copy()
        r[EXTRACT_COL] = format_perturb(row[EXTRACT_COL], level)
        r['perturbation_type'] = 'format'
        r['perturbation_level'] = level
        rows.append(r)

cfb_perturbed = pd.DataFrame(rows)
cfb_perturbed.to_csv(DATA_DIR / "cfb_perturbed.csv", index=False)

print(f"CFB: {len(cfb)} original x 13 variants = {len(cfb_perturbed)} rows")
cfb_perturbed[['Question ID', 'Type of question', 'Answer', 'perturbation_type', 'perturbation_level']].head(14)

CFB: 330 original x 13 variants = 4290 rows


,Question ID,Type of question,Answer,perturbation_type,perturbation_level
0,Q1,LR,According to the company's statement for fisca...,none,0
0,Q1,LR,According to the company's statement for fisca...,numeric,1
0,Q1,LR,According to the company's statement for fisca...,missing,1
0,Q1,LR,According to the company's statement for fisca...,entity,1
0,Q1,LR,According to the company's statement for fisca...,format,1
0,Q1,LR,According to the company's statement for fisca...,numeric,2
0,Q1,LR,According to the company's statement for fisca...,missing,2
0,Q1,LR,According to the company's statement for fisca...,entity,2
0,Q1,LR,According to the company's statement for fisca...,format,2
0,Q1,LR,According to the company's statement for fisca...,numeric,3


In [9]:
# ── Sanity checks ─────────────────────────────────────────────────────────────

for name, df_p, gt_col in [
    ("ESGenius", esg_perturbed, 'answer'),
    ("CFB",      cfb_perturbed, 'Answer'),
]:
    print(f"\n{'─'*50}")
    print(f"{name}  ({len(df_p)} rows)")
    print(df_p.groupby(['perturbation_type', 'perturbation_level']).size()
              .rename('count').to_string())

    # Ground truth must be identical to the unperturbed rows
    original_gt = df_p.loc[df_p['perturbation_level'] == 0, gt_col].reset_index(drop=True)
    for p_type in df_p['perturbation_type'].unique():
        if p_type == 'none':
            continue
        for level in [1, 2, 3]:
            subset_gt = (df_p
                         .loc[(df_p['perturbation_type'] == p_type) &
                              (df_p['perturbation_level'] == level), gt_col]
                         .reset_index(drop=True))
            assert original_gt.equals(subset_gt), \
                f"Ground truth mismatch in {name} {p_type} level {level}!"

print("\nAll ground-truth columns are intact across every perturbation variant.")


──────────────────────────────────────────────────
ESGenius  (14768 rows)
perturbation_type  perturbation_level
format             1                     1136
                   2                     1136
                   3                     1136
label              1                     1136
                   2                     1136
                   3                     1136
missing            1                     1136
                   2                     1136
                   3                     1136
none               0                     1136
numeric            1                     1136
                   2                     1136
                   3                     1136

──────────────────────────────────────────────────
CFB  (4290 rows)
perturbation_type  perturbation_level
entity             1                     330
                   2                     330
                   3                     330
format             1                     330
  